# 3.1 · 探索性数据分析 / Exploratory Data Analysis (EDA)

> **课程定位 / Where this fits**
> 第 1 课，**Part 3 · EDA 与数据预处理**。
> Lesson 1, **Part 3 · EDA & Preprocessing**.
>
> 拿到一份数据，**建模之前的第一件事永远是 EDA**——先把数据"看明白"：有多少行多少列、什么类型、哪里有缺失/异常、各特征长什么样、它们和目标有没有关系。EDA 做得好，后面的清洗、特征工程、建模才有方向。
> When you get a dataset, **the first thing before any modeling is always EDA** — understand the data: how many rows/columns, what types, where the missing/outlier values are, how each feature is distributed, and whether it relates to the target. Good EDA gives direction to everything that follows.
>
> 💼 **实战/面试视角**：几乎所有数据岗的实战题、case 面都从 EDA 开始；"拿到一份新数据你会先做什么"是高频开放题。
> 💼 **Practical/interview angle:** almost every take-home and case interview starts with EDA; "what's the first thing you'd do with a new dataset?" is a very common open question.

> 💡 **面试相关 / Interview-relevant**
> - "拿到新数据先做什么 / EDA 流程"（出镜率 ★★★★★）
> - "右偏数据怎么处理（log 变换）"（★★★★）
> - "怎么判断特征和目标有没有关系"（★★★★）
> - "相关 ≠ 因果 / 怎么发现交互效应"（★★★★）
> - "EDA 阶段要不要看测试集（泄漏）"（★★★★）

---

## 学习目标 / Learning Objectives

1. 掌握一套**可复用的 EDA 流程**：结构 → 质量 → 分布 → 与目标的关系。
   Master a **reusable EDA workflow**: structure → quality → distributions → relationship with the target.
2. 用**数据质量报告**一眼揪出常量列、ID 列、高缺失列。
   Use a **data-quality report** to spot constant columns, ID-like columns, high-missing columns.
3. 识别**偏态**并用 log 变换处理。
   Recognize **skew** and handle it with a log transform.
4. 用图和**统计检验**确认"特征↔目标"的关系（接 Part 2）。
   Confirm feature↔target relationships with plots and **statistical tests** (continuing Part 2).
5. 理解 EDA 阶段的**泄漏意识**：探索只在训练集上做。
   Understand **leakage-awareness** in EDA: explore only on the training set.

## 目录 / TOC
1. [先建直觉：EDA 到底在找什么](#1)
2. [🚢 数据 + 结构与质量报告 ⭐](#2)
3. [单变量：分布与偏态 ⭐](#3)
4. [特征 vs 目标：哪些有用 ⭐](#4)
5. [相关与交互效应 ⭐](#5)
6. [用统计检验确认假设](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 先建直觉：EDA 到底在找什么 / Intuition First

EDA 不是漫无目的地画图，而是带着**四个问题**去"审讯"数据：
EDA isn't aimless plotting; it interrogates the data with **four questions**:

1. **结构**：多少行、多少列、每列什么类型？（先搞清楚手上有什么）
   **Structure:** how many rows/columns, what type is each? (Know what you have.)
2. **质量**：哪里有缺失、重复、常量列、像 ID 的列、明显异常？（找出要清洗的地方）
   **Quality:** where are missing values, duplicates, constant columns, ID-like columns, obvious anomalies? (Find what needs cleaning.)
3. **分布**：每个特征长什么样？是否偏态、是否需要变换？（决定预处理动作）
   **Distribution:** how is each feature shaped — skewed? in need of transformation? (Decide preprocessing.)
4. **关系**：特征和目标有没有关系？特征之间呢？有没有交互？（决定哪些特征值得用）
   **Relationships:** do features relate to the target, and to each other? any interactions? (Decide which features matter.)

> ⚠️ **一个常被忽略的实战要点**：严格来说，EDA 应该**只在训练集上做**。如果你先看了测试集再据此设计特征，就把测试集的信息"偷"进了模型——这叫**数据泄漏(data leakage, 3.9)**。
> ⚠️ **An often-missed practical point:** strictly, EDA should be done **only on the training set**. Peeking at the test set to design features leaks test information into your model — **data leakage (3.9)**.


<a id="2"></a>
## 2. 数据 + 结构与质量报告 ⭐ / Data, Structure & Quality Report

我们用 **Titanic**（泰坦尼克乘客数据，seaborn 内置）：每行一名乘客，目标 `survived`(是否生还)。它混合了数值、类别特征，还带缺失值，是 EDA 教学的经典脏数据。
We use **Titanic** (passenger data, built into seaborn): one row per passenger, target `survived`. It mixes numeric and categorical features and has missing values — the classic messy dataset for EDA.

第一步永远是 `shape` + `head`：看清规模和长相。
The first step is always `shape` + `head`: see the size and a sample.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

df = sns.load_dataset("titanic")
print(f"shape 形状: {df.shape}  (行 rows × 列 columns)")
df.head(3)


**数据质量报告**：与其一列列手看，不如写一个**可复用的函数**一次性汇总每列的类型、缺失率、唯一值数，并自动标记可疑列。这是实战里拿到任何新数据都会先跑的一步。
**A data-quality report:** rather than eyeballing column by column, write a **reusable function** that summarizes each column's type, missing rate, and unique count in one shot, auto-flagging suspicious columns. In practice this is the first thing you run on any new dataset.


In [ ]:
def quality_report(df):
    rep = pd.DataFrame({
        "dtype": df.dtypes.astype(str),                 # 每列的数据类型
        "n_missing": df.isna().sum(),                   # 缺失值个数 (isna() 标记缺失, sum 计数)
        "pct_missing": (df.isna().mean()*100).round(1), # 缺失比例% (mean 对布尔=比例)
        "n_unique": df.nunique(),                       # 不同取值的个数
        "pct_unique": (df.nunique()/len(df)*100).round(1),  # 唯一值占比%
    })
    # 自动标记可疑列 / auto-flag suspicious columns
    rep["flag"] = ""
    rep.loc[rep.n_unique == 1, "flag"] += "常量constant;"          # 只有1个值→对模型无用
    rep.loc[rep.n_unique == len(df), "flag"] += "全唯一(像ID)ID-like;"  # 每行不同→可能是ID, 别当特征
    rep.loc[rep.pct_missing > 50, "flag"] += "高缺失high-missing;"      # 缺失过半→考虑丢弃
    return rep

quality_report(df)


In [ ]:
# 重复行检查: 完全相同的行常是数据录入/合并错误 / duplicate rows are often data errors
print(f"完全重复的行数 fully-duplicated rows: {df.duplicated().sum()}")
print("实战中还应按'业务主键'(如用户ID+时间)查重; Titanic 没有显式 ID 列")
print("In practice also check duplicates on a business key (e.g. user_id+timestamp).")


<a id="3"></a>
## 3. 单变量：分布与偏态 ⭐ / Univariate: Distributions & Skew

逐个看每个特征的分布，重点关注**偏态(skew)**。很多数值特征（价格、收入、计数）是**右偏**的——少数极大值把分布拖出一条长尾。右偏会伤害线性模型和依赖距离的模型，**标准动作是取 log**（`log1p` = $\log(1+x)$，能处理 0）把它拉回接近对称。
Look at each feature's distribution one at a time, watching for **skew**. Many numeric features (price, income, counts) are **right-skewed** — a few huge values drag a long tail. Skew hurts linear and distance-based models; the **standard move is a log transform** (`log1p` = $\log(1+x)$, which handles 0) to pull it back toward symmetry.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

# 数值 age: 有缺失, 单峰、略右偏 / numeric, slightly right-skewed
sns.histplot(df["age"].dropna(), bins=30, kde=True, ax=axes[0,0])
axes[0,0].set_title(f"age (偏度 skew={st.skew(df['age'].dropna()):.2f})")

# 数值 fare: 极右偏(少数高价票拖长尾) — 注意! / very right-skewed
sns.histplot(df["fare"], bins=40, ax=axes[0,1])
axes[0,1].set_title(f"fare (skew={st.skew(df['fare']):.2f}) — 极右偏 very skewed")

# log1p 变换后: 长尾被压缩, 分布接近对称(右偏数据的标准处理) / log fixes skew
sns.histplot(np.log1p(df["fare"]), bins=40, ax=axes[0,2])
axes[0,2].set_title("log1p(fare) — 接近对称了 nearly symmetric")

# 类别 pclass: 用 countplot 数每个取值的频数 / categorical counts
sns.countplot(data=df, x="pclass", ax=axes[1,0]); axes[1,0].set_title("pclass (三等舱最多 mostly 3rd)")
sns.countplot(data=df, x="embarked", ax=axes[1,1]); axes[1,1].set_title("embarked 登船港口")

# 目标 survived: 检查类别是否平衡(影响后续指标选择, 见 3.11/5.1) / target balance
surv = df["survived"].value_counts(normalize=True)   # normalize=True → 比例
axes[1,2].bar(["died(0)", "survived(1)"], surv.values, color=["#d62728","#2ca02c"])
axes[1,2].set_title(f"survived: {surv[1]:.0%} 生还 survived (略不平衡 mildly imbalanced)")
plt.tight_layout(); plt.show()


<a id="4"></a>
## 4. 特征 vs 目标：哪些有用 ⭐ / Features vs Target

这是 EDA 最有价值的一步：**看每个特征和目标的关系**，初步判断哪些特征有预测力。
The most valuable EDA step: **look at each feature against the target** to gauge predictive power.
- **类别特征**：看各类别下的目标均值（这里就是各组的"生还率"）。
  **Categorical:** the target mean within each category (here, the survival rate per group).
- **数值特征**：按目标分组看分布（箱线图最直观）。
  **Numeric:** the distribution split by target (a boxplot is most intuitive).


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

# 类别特征 vs 目标: barplot 的 y 是均值, 这里就是各组生还率 / barplot y = mean = survival rate
for ax, col in zip(axes[0], ["sex", "pclass", "embarked"]):
    sns.barplot(data=df, x=col, y="survived", ax=ax)
    ax.set_title(f"survival rate by {col}"); ax.set_ylim(0, 1)

# 数值特征 vs 目标: 按 survived 分组看分布 / numeric split by target
sns.boxplot(data=df, x="survived", y="age", ax=axes[1,0]); axes[1,0].set_title("age by survived")
sns.boxplot(data=df, x="survived", y="fare", ax=axes[1,1])
axes[1,1].set_yscale("log"); axes[1,1].set_title("fare by survived (log y轴)")

# 派生特征 family_size: EDA 常顺手造新特征看看 / engineer a feature on the fly
df2 = df.assign(family_size=df.sibsp + df.parch + 1)   # 兄弟姐妹+父母子女+自己
sns.barplot(data=df2, x="family_size", y="survived", ax=axes[1,2])
axes[1,2].set_title("survival by family_size (非单调! non-monotonic)")
plt.tight_layout(); plt.show()
print("发现: sex/pclass/fare 与生还强相关; family_size 呈非单调(中等家庭生还率最高)")
print("Findings: sex/pclass/fare strongly relate to survival; family_size is non-monotonic.")


<a id="5"></a>
## 5. 相关与交互效应 ⭐ / Correlation & Interactions

**相关矩阵**快速看数值特征两两之间、以及和目标的线性关系。但要牢记两条实战铁律：
A **correlation matrix** quickly shows pairwise linear relationships among numeric features and with the target. Keep two practical rules in mind:
- **相关 ≠ 因果**：高相关只说明同步变化，不代表谁导致谁。
  **Correlation ≠ causation:** high correlation means co-movement, not that one causes the other.
- 相关系数只抓**线性**关系；非线性关系（如 family_size）可能相关系数很低却很有用。
  Correlation only catches **linear** relationships; a nonlinear one (like family_size) can have low correlation yet be useful.

**交互效应(interaction)**：两个特征**合起来**的影响超过各自单独之和。用透视表+热力图最容易看出来。
**Interaction:** two features **together** have an effect beyond their individual sum. A pivot table + heatmap reveals it best.


In [ ]:
num = df[["survived","pclass","age","sibsp","parch","fare"]]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
# .corr() 算两两皮尔逊相关系数; annot 显示数值, center=0 让正负对称着色
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0, ax=axes[0])
axes[0].set_title("相关矩阵 correlation matrix")

# 交互: 舱等 × 性别 → 生还率; pivot_table 按两个维度分组求均值 / interaction via pivot
pivot = df.pivot_table(values="survived", index="pclass", columns="sex", aggfunc="mean", observed=True)
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Greens", vmin=0, vmax=1, ax=axes[1])
axes[1].set_title("生还率 survival: pclass × sex (强交互 strong interaction!)")
plt.tight_layout(); plt.show()
print("一等舱女性生还率~97%, 三等舱男性~13% → 性别的影响在不同舱等下差异巨大(交互)")
print("First-class women ~97% vs third-class men ~13% → sex's effect varies by class (interaction).")


<a id="6"></a>
## 6. 用统计检验确认假设 / Confirming Hypotheses with Tests

EDA 图上看到的差异，**有可能只是随机噪声**。用 Part 2 学的假设检验把"看起来有差异"升级成"统计上显著"。这里检验"性别是否影响生还"——两个类别变量的关系用**卡方检验**。
A difference seen in an EDA plot **might just be noise**. Use the hypothesis tests from Part 2 to upgrade "looks different" into "statistically significant". Here we test "does sex affect survival" — the relationship between two categorical variables uses a **chi-square test**.


In [ ]:
# 列联表: 性别 × 生还 的交叉计数 / contingency table of sex vs survived
ct = pd.crosstab(df["sex"], df["survived"])
chi2, p, dof, _ = st.chi2_contingency(ct)   # 卡方独立性检验 / chi-square test of independence
print("列联表 contingency table:\n", ct)
print(f"\n性别 × 生还 卡方检验: chi2={chi2:.1f}, p={p:.2e}")
print("→ p 远小于 0.001, 性别差异在统计上极显著, 不是噪声")
print("→ p ≪ 0.001: the sex difference is highly significant, not noise.")


<a id="7"></a>
## 7. 小结 / Summary

```
EDA 四问: 结构(行列/类型) → 质量(缺失/重复/常量/ID) → 分布(偏态/变换) → 关系(特征↔目标, 交互)
质量报告: 写可复用函数一次性汇总+标记可疑列(常量/ID/高缺失)
偏态: 右偏数值(价格/收入)用 log1p 拉回对称
特征↔目标: 类别看分组目标均值, 数值看分组分布(箱线图)
相关矩阵: 只抓线性关系; 相关≠因果; 透视表+热力图找交互
统计检验(Part 2)把"看起来有差异"确认成"统计显著"
泄漏意识: EDA 只在训练集做
```

### 💡 面试速查 / Interview cheat-sheet
1. **拿到新数据先 EDA**：结构→质量→分布→关系，四步走。
   New data → EDA first: structure → quality → distribution → relationships.
2. **右偏数据用 log 变换**（log1p 可处理 0）。
   Right-skewed data → log transform (log1p handles 0).
3. **相关≠因果**，且相关系数只抓线性；非线性/交互要专门看。
   Correlation ≠ causation, and it's linear-only; check nonlinearity/interactions separately.
4. **类别平衡**要在 EDA 就发现（影响后续指标与重采样，见 3.11/5.1）。
   Spot class imbalance during EDA (affects metrics & resampling later).
5. **EDA 只在训练集做**，否则有泄漏风险（3.9）。
   Do EDA on the training set only, or risk leakage (3.9).

### 下一节 / Next
**3.2 缺失值处理**——EDA 发现了缺失，这一课系统讲怎么处理：删除、各种填补(均值/中位数/KNN/模型)，以及"缺失本身也是信息"。
**3.2 Missing Values** — EDA found gaps; this lesson systematically covers handling them: deletion, imputation (mean/median/KNN/model), and "missingness itself can be informative".
